# AI-PoweredUX

**Module 12 · Lesson 12.10**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Perceived Performance and Streaming
- Loading States: Skeleton, Not Spinner
- Optimistic UI
- Streaming the Reasoning
- Confidence and Citations
- Error and Graceful-Degradation UX

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## Same model, one feels broken and one feels magic

> 💡 **Info**
>
> Here is the uncomfortable truth about AI products: **two teams can ship the exact same model, behind the exact same API, and one of them will be called broken while the other is called magic.** Team A returns the whole answer after a five-second blank screen, shows a raw **“500 Internal Server Error”** when anything breaks, and never tells you how sure the AI is. Team B streams the first word in **200 milliseconds** so you read while it thinks, shows a **skeleton** instead of a spinner, turns every error into “our AI is having a moment, retrying…”, and marks its shaky answers **“verify this”** with a source you can click. Nothing about the backend is different. Everything about how it *feels* is. That gap is **UX** — the last mile that decides whether all the architecture, RAG, agents, and evals from the last eleven modules actually reach a human who trusts them. This lesson builds that last mile: perceived-speed streaming, skeleton loading, optimistic UI, reasoning disclosure, honest confidence and citations, graceful error recovery, and the human-in-the-loop confirmations and thumbs-feedback that close the loop back to your evals — every piece a keyless model you can run with no browser, no provider, and no key.

### 🎯 What you will be able to do after this lesson

- **Build** AI-product UX — a perceived-speed streaming model, a loading-state selector, optimistic updates, reasoning disclosure, a confidence/citation renderer, an error-to-recovery map, and a HITL + feedback loop — as runnable models plus an illustrative streaming stack.

- **Compare** a blank-wait vs a streamed response, and a spinner vs a skeleton, on *perceived* performance.

- **Operate** the reasoning-disclosure pattern, the claim-level citation + verify affordance, and the confirm-before-you-act gate.

- **Evaluate** an AI UX: does it feel fast, tell the user what is happening, recover gracefully, disclose uncertainty honestly, and confirm risky actions?

> 📦 **Info**
>
> ✅ Before you startIn **7.4** you streamed tokens from the API server-side; this lesson is what the *user* sees when you do. In **12.8** you scored quality on live traffic; the thumbs-up/down this lesson captures is where that signal is born. In **12.9** you degraded gracefully during incidents; the error UX here is the user-facing face of that fallback. The **cost** behind the latency you tune comes in Module 13; the **eval set** built from this feedback is built in Lesson 14.4; the human-in-the-loop agent internals were in Lesson 11.10.

> ℹ️ **Info**
>
> 💡 Before any of these: set expectationsThe most under-rated AI UX happens *before* the first token — telling the user what the AI can and cannot do, and how reliably. Both the Microsoft HAX guidelines and the Google People + AI guidebook open with exactly this: an **empty state that shows example prompts** and names the limits builds a correct mental model, so users lean on the good answers and forgive the weak ones. The seven patterns below are what you do *during* and *after* a request; the expectation you set up front is what makes them land.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍽️ **Analogy**
>
> Great AI UX is a **great restaurant, not just a great kitchen**. The kitchen (your model and backend) can be world-class, but the diner only ever experiences the *service*: the bread that arrives the moment you sit down so you are not staring at an empty table (streaming), the waiter who says “the risotto takes twenty minutes” instead of vanishing (loading states), the honest “we are out of the sea bass, may I suggest the trout?” (graceful errors and confidence), and the check brought only when you ask (confirm before acting). **Where it breaks down:** a restaurant serves one table at a time, while your UX serves every user at once — so these patterns have to be built into the product, not improvised per diner.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Only a faster model can feel faster.”** Streaming makes the *same* generation time feel near-instant, because the user reads while the model thinks. Perceived performance is a UX lever, not just a backend one.
> - **“Hide uncertainty so the product looks confident.”** A hallucination styled like a fact destroys trust the moment it is found. Honest confidence and citations build *calibrated* trust — users verify the shaky parts and rely on the solid ones.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that quietly wreck an AI product. **Rendering the model’s chain-of-thought as the answer** — users cannot tell the deliberation from the conclusion; put reasoning in a collapsible, default-closed disclosure. And **showing a raw “500” or auto-executing a destructive action with no confirm** — the first reads as broken, the second is how an agent deletes the wrong records; every error gets a friendly recovery, every risky action gets a confirm.

---

## 🎯 Concept 1: Perceived Performance and Streaming

### Perceived Performance and Streaming

The same request can feel five seconds slow or near-instant depending only on the front-end. Streaming tokens means the user reads from the first token (about 200 milliseconds) while the model keeps generating, so the perceived wait collapses to the time-to-first-token even though the total time is unchanged.

Start with the single most valuable UX decision in AI products: **streaming**. When a model takes a few seconds to generate an answer, you have two choices. Hold everything and reveal the whole answer at the end — the user stares at a **blank screen** for the full generation time. Or **stream** the tokens as they are produced — the first word appears in about **200 milliseconds** and the user starts *reading* while the model is still writing. The total generation time is *identical* in both cases; the backend has not changed at all. What changes is the **perceived** wait, which collapses from the full generation time to the **time-to-first-token** (TTFT). This is why ChatGPT felt revolutionary — not a better model, but the first token on screen in a fraction of a second. The rule: show *something* the instant the user submits, stream from the first token, and always give them a visible **stop** button. Streaming is also about *how* the tokens render, not just when: the model streams **markdown** token by token, so each frame hands the UI syntactically incomplete markdown (an unclosed code fence, a half-built table) that a streaming-markdown renderer must handle gracefully. And as the answer grows past the fold, **auto-scroll** to follow the newest tokens but release that lock the instant the user scrolls up to re-read, with a “jump to latest” control to re-attach. The block races the two front-ends, keyless.

> 👨‍🍳 **Analogy**
>
> It is a **chef sending out courses, not one giant plate at the end**. Two kitchens take the same ninety minutes to cook the same meal. One sends nothing until the whole dinner is plated, so you sit at an empty table for an hour and a half, starving and annoyed. The other brings the bread and the starter within minutes, then the courses as they are ready — you are eating and happy the entire time, even though dinner finishes at the exact same moment. Streaming is coursing the meal: the same total time, a completely different experience of the wait.

A response takes 3 seconds to generate. You switch from a blank wait to streaming tokens. What happens to the TOTAL generation time and the PERCEIVED wait?

**📝 `01_streaming_ttft.py`** - *runnable*

In [ ]:
# The backend is identical. Only the UX changes the PERCEIVED wait.
TOTAL_MS = 3000     # same generation time either way
TOKENS = 150
TTFT_MS = 200       # time to FIRST token when streaming

# BLANK: hold everything, reveal the whole answer at the end.
blank_first_content = TOTAL_MS          # user stares at nothing until it is all done
# STREAMING: first token at TTFT, then the user reads while the model keeps generating.
stream_first_content = TTFT_MS
per_token = (TOTAL_MS - TTFT_MS) / TOKENS

print("Same request, same model, {}ms to generate {} tokens. Two front-ends:".format(TOTAL_MS, TOKENS))
print()
print("  BLANK WAIT:  nothing on screen for {}ms, then the whole answer appears at once.".format(blank_first_content))
print("     -> the user stares at a blank screen for {}ms".format(blank_first_content))
print("  STREAMING:   first token at {}ms, then ~1 token every {:.0f}ms while the user reads.".format(TTFT_MS, per_token))
print("     -> the user starts reading after just {}ms".format(stream_first_content))
print()
speedup = blank_first_content / stream_first_content
print("Perceived wait (time to first content): {}ms -> {}ms = {:.0f}x lower.".format(blank_first_content, stream_first_content, speedup))
print("Total time is UNCHANGED ({}ms). Streaming is a UX lever, not a backend one:".format(TOTAL_MS))
print("the user reads while the model thinks, so the wait feels near-zero. Always show a stop button.")

# Output:
# Same request, same model, 3000ms to generate 150 tokens. Two front-ends:
#
#   BLANK WAIT:  nothing on screen for 3000ms, then the whole answer appears at once.
#      -> the user stares at a blank screen for 3000ms
#   STREAMING:   first token at 200ms, then ~1 token every 19ms while the user reads.
#      -> the user starts reading after just 200ms
#
# Perceived wait (time to first content): 3000ms -> 200ms = 15x lower.
# Total time is UNCHANGED (3000ms). Streaming is a UX lever, not a backend one:
# the user reads while the model thinks, so the wait feels near-zero. Always show a stop button.

- The **same** request generates the same tokens in the same total time under both front-ends.
- **Blank wait**: nothing on screen for the whole generation, so the user stares at a blank screen for the full duration.
- **Streaming**: the first token lands at the TTFT (about 200 milliseconds) and the user reads while the rest streams in.
- Perceived wait (time to first content) drops many-fold while the **total time is unchanged** — streaming is a UX lever, not a backend one.

#### 💡 What just happened

⚡ What just happened?Streaming collapses the PERCEIVED wait to the time-to-first-token because the user reads while the model generates, even though the total generation time is identical. Tradeoff / the framing for the lesson: perceived performance is something you win in the front-end, not only in the backend - and every later step is another way to make an AI app feel fast, honest, and trustworthy. Always pair streaming with a visible stop button.

- Race two response bars for the same three-second generation: a blank-wait bar vs a streaming bar
- The streaming bar shows the first token in about 200 milliseconds and a perceived-wait readout collapsing to the TTFT

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Loading States: Skeleton, Not Spinner

### Loading States: Skeleton, Not Spinner

Match the indicator to the wait. Under 0.1s feels instant; under 1s keeps flow; up to 10s needs a progress indicator; beyond that needs phases and a cancel. A skeleton of the coming content feels about 40 percent shorter than a blank spinner.

Before the first token even arrives, you have a wait to fill — and not all waits are equal. Nielsen’s three response-time limits still govern: under about **0.1 second** feels instant (show nothing), under about **1 second** keeps the user in flow (keep it subtle), up to about **10 seconds** holds attention only if you show progress, and beyond that you need **phased messages**, an estimate, and a **cancel** button. The single best progress indicator for an AI response is a **skeleton screen** — grey shimmer placeholders in the rough shape of the content that is coming, a few lines at decreasing widths — because it sets expectations and shows structure. A blank **spinner** past a second reads as *stuck*; the identical wait behind a skeleton feels roughly **40 percent shorter**, because the user can see that something specific is on its way. Match the indicator to the wait, and prefer a skeleton to a spinner. The block picks the right indicator per wait, keyless.

> 📝 **Analogy**
>
> It is a **menu versus a locked kitchen door**. You walk into two restaurants that will both take fifteen minutes to cook. One seats you facing a blank, closed kitchen door — you have no idea if anyone is even back there, and every minute drags (the spinner). The other hands you a menu with photos of exactly what is coming — you can see the shape of your meal, so the same fifteen minutes feels shorter and calmer (the skeleton). Showing the structure of what is on its way is what turns an anxious wait into a patient one.

A request will take about 4 seconds. Which loading indicator is right?

**📝 `02_loading_states.py`** - *runnable*

In [ ]:
# Match the indicator to the wait. Nielsen's three response-time limits (0.1s / 1s / 10s).
def indicator(expected_ms):
    if expected_ms < 100:    return "nothing (feels instant)"
    if expected_ms < 1000:   return "a subtle hint (keep the flow, no spinner needed)"
    if expected_ms < 10000:  return "a SKELETON of the coming content (hold attention)"
    return "PHASED messages + an estimate + a Cancel button"

print("The right loading indicator by expected wait:")
for ms in [80, 600, 4000, 25000]:
    print("  {:>6}ms -> {}".format(ms, indicator(ms)))
print()

# Skeleton vs spinner: a skeleton sets expectations, so the SAME wait FEELS ~40% shorter.
actual_ms = 4000
SKELETON_FACTOR = 0.6    # perceived wait is ~60% of actual with a skeleton (~40% cut)
spinner_perceived = actual_ms
skeleton_perceived = int(actual_ms * SKELETON_FACTOR)
print("Skeleton vs spinner for the same {}ms wait:".format(actual_ms))
print("  blank panel + spinner: feels like {}ms (opaque, reads as broken past ~1s)".format(spinner_perceived))
print("  skeleton screen:       feels like {}ms (~40% lower - it shows the shape of what is coming)".format(skeleton_perceived))
print()
print("A spinner past ~1s reads as stuck; a skeleton (a few grey lines at decreasing widths) reassures.")

# Output:
# The right loading indicator by expected wait:
#       80ms -> nothing (feels instant)
#      600ms -> a subtle hint (keep the flow, no spinner needed)
#     4000ms -> a SKELETON of the coming content (hold attention)
#    25000ms -> PHASED messages + an estimate + a Cancel button
#
# Skeleton vs spinner for the same 4000ms wait:
#   blank panel + spinner: feels like 4000ms (opaque, reads as broken past ~1s)
#   skeleton screen:       feels like 2400ms (~40% lower - it shows the shape of what is coming)
#
# A spinner past ~1s reads as stuck; a skeleton (a few grey lines at decreasing widths) reassures.

- The **indicator selector** maps the expected wait to the right UI by Nielsen’s limits: nothing under 0.1s, subtle under 1s, a **skeleton** up to 10s, **phased + cancel** beyond.
- A **skeleton** for a 4-second wait feels about **40 percent shorter** than a blank spinner for the same wait.
- A spinner past about a second reads as stuck; a skeleton (grey lines at decreasing widths) reassures by showing the shape of what is coming.
- The lesson: never show a bare spinner for a long AI wait — set expectations with a skeleton, and phase-plus-cancel the very long ones.

#### 💡 What just happened

⚡ What just happened?Match the loading indicator to the wait (Nielsen’s 0.1s / 1s / 10s limits), and prefer a skeleton to a spinner because it sets expectations and feels about 40 percent shorter. Tradeoff: a skeleton costs a little markup that mirrors your content’s shape, in exchange for a calmer, shorter-feeling wait. Next: making the round-trip itself feel instant.

- A wait-duration slider lights the right indicator by Nielsen threshold (none / subtle / skeleton / phased+cancel)
- A skeleton-vs-spinner toggle shows the same wait feeling about 40 percent shorter with a skeleton

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Optimistic UI

### Optimistic UI

Echo the user’s action before the round-trip completes: their message appears instantly, then you reconcile with the server on success or roll back on failure. This eliminates the ‘did my message even send?’ gap of naive fetch-then-render.

Streaming handles the model’s response; **optimistic UI** handles the user’s own action. When a user hits send, the naive approach fires a request and waits for the server before showing anything — so for the whole round-trip the thread looks unchanged and the user wonders *“did my message even send?”* The optimistic approach echoes the action **immediately**: the user’s message appears in the thread at once (perceived latency zero), and *then* you **reconcile** with the server — keep it on success, or **roll back** on failure (remove the message, restore what they typed, show a clear error). The optimistic bet is that the request will usually succeed, so you show success first and handle the rare failure gracefully. React’s `useOptimistic` and the Vercel AI SDK `useChat` hook do this user-message insert for you. The one real cost is that you must handle the rollback path. The block runs both the success and the failure, keyless.

> 🛒 **Analogy**
>
> It is a **cashier who bags your groceries while the card is still processing**. A slow shop makes you stand and wait for the terminal to beep before anyone touches your items — every second of the round-trip is dead time. A good cashier bags everything *while* the payment clears, because it almost always clears; you are already walking out as it confirms. And on the rare decline, they simply un-bag it and ask for another card (the rollback). Optimistic UI is bagging while the card processes: act on the likely outcome now, handle the rare failure cleanly.

A user sends a chat message. In an optimistic UI, when does it appear in the thread, and what happens if the server call fails?

**📝 `03_optimistic_ui.py`** - *runnable*

In [ ]:
# Optimistic UI: echo the user's action BEFORE the round-trip, then reconcile or roll back.
SERVER_MS = 800    # how long the real round-trip takes

def send(message, server_ok):
    log = []
    log.append("  t=0ms    user hits send -> message appears in the thread INSTANTLY (optimistic)")
    log.append("  t=0ms    perceived send-latency for the user: 0ms")
    log.append("  t={}ms  server responds...".format(SERVER_MS))
    if server_ok:
        log.append("  t={}ms  RECONCILE: server confirmed -> keep the message, mark it delivered".format(SERVER_MS))
    else:
        log.append("  t={}ms  ROLLBACK: server failed -> remove the message, restore the input, show a clear error".format(SERVER_MS))
    return log

print("Optimistic send that SUCCEEDS:")
for line in send("what is MCP?", True): print(line)
print()
print("Optimistic send that FAILS:")
for line in send("what is MCP?", False): print(line)
print()
print("Naive fetch-then-render would show nothing for {}ms ('did my message even send?').".format(SERVER_MS))
print("Optimistic UI makes the send feel instant; the ONLY cost is you must handle the rollback path.")

# Output:
# Optimistic send that SUCCEEDS:
#   t=0ms    user hits send -> message appears in the thread INSTANTLY (optimistic)
#   t=0ms    perceived send-latency for the user: 0ms
#   t=800ms  server responds...
#   t=800ms  RECONCILE: server confirmed -> keep the message, mark it delivered
#
# Optimistic send that FAILS:
#   t=0ms    user hits send -> message appears in the thread INSTANTLY (optimistic)
#   t=0ms    perceived send-latency for the user: 0ms
#   t=800ms  server responds...
#   t=800ms  ROLLBACK: server failed -> remove the message, restore the input, show a clear error
#
# Naive fetch-then-render would show nothing for 800ms ('did my message even send?').
# Optimistic UI makes the send feel instant; the ONLY cost is you must handle the rollback path.

- On **send**, the user’s message appears in the thread at **t=0** — perceived send-latency is zero, before any round-trip.
- The **success** path **reconciles** at the server response: keep the message, mark it delivered.
- The **failure** path **rolls back**: remove the optimistic message, restore the input, and show a clear error.
- Naive fetch-then-render would show nothing for the whole round-trip (“did it send?”); the only cost of optimism is handling that rollback.

#### 💡 What just happened

⚡ What just happened?Optimistic UI echoes the user’s action instantly and then reconciles with the server (keep on success, roll back on failure), eliminating the ‘did it send?’ gap. Tradeoff: you bet on the common case (success) and must handle the rare rollback cleanly, in exchange for a send that feels instant. Next: how to show the model’s reasoning without confusing it for the answer.

- Send a message: it appears instantly (optimistic), a server clock runs to 800 milliseconds
- Then it reconciles (keep, emerald) or, on a forced failure, rolls back (rose) and restores the input

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Streaming the Reasoning

### Streaming the Reasoning

A thinking model emits a long reasoning trace before its answer. Dumped as the answer, users mistake it for the result; hidden, power users lose it; delayed, you pay a silence tax. The pattern is a collapsible ‘Thinking’ disclosure, default closed, above the answer.

Modern reasoning models produce a long **chain-of-thought** before the final answer, and how you render it makes or breaks the experience. Three wrong ways: **dump it as the answer**, and users cannot tell the model’s scratch-work from its conclusion; **hide it entirely**, and the power users who want to inspect the reasoning lose it; **wait** for the whole reasoning trace to finish before showing anything, and you pay a **silence tax** — a long blank screen while the model deliberates privately. The pattern that works is a **collapsible “Thinking” disclosure, default closed**, rendered *above* the answer and streamed live so the user sees motion without mistaking the deliberation for the result. The answer is front and center; the reasoning is one click away for anyone who wants to audit it. The block compares the three renderings and the timing tradeoff, keyless.

> 🧮 **Analogy**
>
> It is a **mathematician’s scratch paper versus the boxed final answer**. When a mathematician hands you a proof, they do not read you every crossed-out attempt and dead end as if it were the theorem — that would be baffling. They give you the clean result, with the working *available* on a separate page if you want to check it. Nor do they shred the scratch paper (a careful reader may want it). The collapsible “Thinking” section is exactly that separate page: the answer up front, the working folded away but there.

A reasoning model produces a 620-token thinking trace and a 90-token answer. Where should the thinking go?

**📝 `04_reasoning_disclosure.py`** - *runnable*

In [ ]:
# A thinking model emits a long reasoning trace, then a short answer. Render it THREE ways.
reasoning_tokens = 620
answer_tokens = 90

options = {
    "dump as answer":       {"shown": reasoning_tokens + answer_tokens, "verdict": "BAD: user mistakes the chain-of-thought for the answer"},
    "hidden entirely":      {"shown": answer_tokens, "verdict": "power users lose the reasoning they wanted to inspect"},
    "collapsible, default closed": {"shown": answer_tokens, "verdict": "GOOD: answer up front, reasoning one click away"},
}
print("Rendering {} reasoning tokens + a {}-token answer:".format(reasoning_tokens, answer_tokens))
for name, o in options.items():
    print("  {:<28} user sees {:>3} tokens first  -> {}".format(name, o["shown"], o["verdict"]))
print()

# The silence-tax vs premature-commitment tradeoff.
print("Timing tradeoff:")
print("  wait for all {} reasoning tokens before showing anything -> a long BLANK (the silence tax)".format(reasoning_tokens))
print("  stream the answer with a live, collapsible 'Thinking' panel -> motion immediately, no confusion")
print()
print("Pattern: put reasoning in a collapsible 'Thinking' disclosure, default CLOSED, above the answer.")

# Output:
# Rendering 620 reasoning tokens + a 90-token answer:
#   dump as answer               user sees 710 tokens first  -> BAD: user mistakes the chain-of-thought for the answer
#   hidden entirely              user sees  90 tokens first  -> power users lose the reasoning they wanted to inspect
#   collapsible, default closed  user sees  90 tokens first  -> GOOD: answer up front, reasoning one click away
#
# Timing tradeoff:
#   wait for all 620 reasoning tokens before showing anything -> a long BLANK (the silence tax)
#   stream the answer with a live, collapsible 'Thinking' panel -> motion immediately, no confusion
#
# Pattern: put reasoning in a collapsible 'Thinking' disclosure, default CLOSED, above the answer.

- **Dump as answer**: the user sees all 710 tokens first and **mistakes the reasoning for the answer** — the bad default.
- **Hidden**: the user sees only the 90-token answer, but power users lose the reasoning they wanted to inspect.
- **Collapsible, default closed**: the answer is up front and the reasoning is **one click away** — the good pattern.
- The timing tradeoff: waiting for all reasoning tokens is a **silence tax** (a long blank); streaming the answer with a live “Thinking” panel shows motion immediately.

#### 💡 What just happened

⚡ What just happened?Render a thinking model’s reasoning in a collapsible ‘Thinking’ disclosure, default closed, above the answer - not dumped as the answer, not hidden, not delayed into a silence tax. Tradeoff: the disclosure is a little extra UI, in exchange for an answer users trust and reasoning they can audit. Next: how to signal how sure the AI is, and where its facts came from.

- A response with a long reasoning trace and a short answer; toggle dump-as-answer / hidden / collapsible
- Stream the collapsible 'Thinking' section live above the answer, default closed

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Confidence and Citations

### Confidence and Citations

Attach citations to individual claims, not paragraphs; preview on hover and deep-link. Surface a confidence signal and a ‘verify this’ affordance on weak answers. A single unsourced claim styled like a fact is the trust killer; honest disclosure builds calibrated trust.

The hardest UX problem in AI is **honesty about uncertainty**. The failure mode is subtle: an answer that is mostly well-sourced but contains *one unsupported claim styled exactly like the sourced ones*. The user absorbs it as fact, and when they later discover it was wrong, they stop trusting the *entire* answer — and your product. The fix is not to hide doubt but to **disclose it precisely**. Attach citations to **individual claims, not whole paragraphs**, so a reader can check each one; preview the source on hover and **deep-link** into the exact passage. Surface a **confidence** or citation-strength signal, and when the answer is weak — a claim without a source, or low relevance — show a **“verify this”** affordance and flag the unsupported claim. This builds *calibrated* trust: users lean on the well-cited parts and double-check the shaky ones. Honest disclosure makes the product *more* trustworthy, not less. The block scores an answer’s claims, keyless.

> 📜 **Analogy**
>
> It is a **footnoted essay versus a claim shouted from a bar stool**. The same sentence lands completely differently depending on whether you can check it. In a good essay, every factual claim carries a footnote you can follow to the source, and the one assertion the author is unsure about is explicitly hedged — so you trust the whole thing *more*, because you can verify it. The bar-stool version states everything with equal breezy confidence, including the one thing that is flat wrong, and once you catch that you believe none of it. Claim-level citations and an honest “verify this” are the footnotes.

An answer has two well-cited claims and one unsourced claim styled just like a fact. What should the UI do?

**📝 `05_confidence_citations.py`** - *runnable*

In [ ]:
# Attach citations to CLAIMS, not paragraphs. The confidence tier follows the support.
claims = [
    {"text": "MCP is an open standard", "source": "Anthropic docs", "relevance": 0.95},
    {"text": "It uses JSON-RPC over stdio or HTTP", "source": "MCP spec", "relevance": 0.88},
    {"text": "It was first shipped in 2019", "source": None, "relevance": 0.0},   # unsupported, styled like a fact
]
supported = [c for c in claims if c["source"]]
coverage = len(supported) / len(claims)
min_rel = min((c["relevance"] for c in supported), default=0.0)

def tier(coverage, min_rel):
    if coverage < 1.0:      return "LOW - verify this answer"   # ANY unsupported claim caps it
    if min_rel >= 0.9:      return "HIGH"
    if min_rel >= 0.7:      return "MODERATE"
    return "LOW - verify this answer"

print("Answer, claim by claim:")
for c in claims:
    if c["source"]:
        print("  [cited]      {:<38} {} ({:.0f}% relevant)".format(c["text"], c["source"], c["relevance"] * 100))
    else:
        print("  [UNSOURCED]  {:<38} <- styled like a fact, but no source".format(c["text"]))
print()
t = tier(coverage, min_rel)
print("Citation coverage: {}/{} claims -> confidence tier: {}".format(len(supported), len(claims), t))
if "LOW" in t:
    print("Show a 'verify this' disclaimer and flag the unsourced claim; do NOT hide the uncertainty.")
print("One unsourced claim styled like a sourced one is what erodes trust - honest disclosure builds it.")

# Output:
# Answer, claim by claim:
#   [cited]      MCP is an open standard                Anthropic docs (95% relevant)
#   [cited]      It uses JSON-RPC over stdio or HTTP    MCP spec (88% relevant)
#   [UNSOURCED]  It was first shipped in 2019           <- styled like a fact, but no source
#
# Citation coverage: 2/3 claims -> confidence tier: LOW - verify this answer
# Show a 'verify this' disclaimer and flag the unsourced claim; do NOT hide the uncertainty.
# One unsourced claim styled like a sourced one is what erodes trust - honest disclosure builds it.

- Each **claim** is shown with its own citation and relevance — or flagged **[UNSOURCED]** when it is styled like a fact but has no source.
- The **confidence tier** follows the support: any unsourced claim drops the answer to **LOW — verify this**.
- On a low tier, the UI shows a **“verify this” disclaimer** and flags the specific unsupported claim — it does *not* hide the uncertainty.
- The lesson: one unsourced claim styled like a sourced one is what erodes trust; claim-level citations plus honest confidence build it back.

#### 💡 What just happened

⚡ What just happened?Attach citations to individual claims (not paragraphs), surface a confidence signal and a ‘verify this’ affordance on weak answers, and flag any unsupported claim rather than hiding it. Tradeoff: honesty about a shaky answer feels risky but earns calibrated trust; a hidden hallucination styled like a fact is what actually loses users. Next: what the user sees when something breaks.

- An answer of three claims; toggle each claim's support on or off
- Watch the confidence tier (high/mod/low) and the 'verify this' flag + claim-level source pills update

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Error and Graceful-Degradation UX

### Error and Graceful-Degradation UX

No user should ever see ‘500 Internal Server Error’. Map every failure to a friendly message plus a recovery action - auto-retry transient errors, redirect auth and billing, suggest simplifying on a timeout - and degrade to a cached or smaller answer rather than showing nothing.

Everything breaks eventually, and the difference between a trusted product and an abandoned one is what the user sees when it does. The rule is absolute: **no user ever sees a raw error code**. Map every failure to a friendly **title**, a plain-language **message**, and a **recovery action**. Transient errors (**429** rate limit, **500** server, **503** overload) get an **auto-retry** with a countdown, so they often heal with no user action at all. Auth (**401**) redirects to log in; billing (**402**) shows upgrade options; a **timeout** suggests trying a simpler question; a network drop retries on reconnect. And the key AI-specific move is **graceful degradation**: when the fresh answer fails, **fall back** to a cached answer or a smaller model rather than showing nothing — a degraded-but-honest answer beats a blank error every time. On a mid-stream disconnect, keep the **partial** streamed text rather than wiping it. The block maps errors to recovery, keyless.

> 🍴 **Analogy**
>
> It is a **good waiter when the kitchen is slammed**. A bad restaurant lets you sit with an empty table and no word, or a server blurts “the kitchen crashed” and walks off (the raw 500). A good waiter comes over with a plan: “the grill is backed up about ten minutes — can I bring you the soup now while you wait?” (auto-retry with a fallback). You are never left confused or empty-handed; there is always a friendly message and a next step. Error UX is being that waiter: every failure gets a human explanation and a way forward.

Your AI service returns a 500 in the middle of a request. What should the user see?

**📝 `06_error_ux.py`** - *runnable*

In [ ]:
# No user ever sees a raw code. Map every failure to a friendly message + a recovery action.
ERRORS = {
    401: {"title": "Session expired",     "action": "redirect to log in"},
    402: {"title": "Usage limit reached", "action": "show upgrade options"},
    429: {"title": "Too many requests",   "action": "auto-retry in 5s (countdown)"},
    500: {"title": "Our AI hit a snag",   "action": "auto-retry in 3s, else fall back to a cached answer"},
    503: {"title": "High demand",         "action": "queue the request, auto-retry in 10s"},
    "timeout": {"title": "Taking a while","action": "offer 'simplify the question'"},
    "network": {"title": "You are offline","action": "auto-retry on reconnect"},
}
def show(code):
    e = ERRORS.get(code, ERRORS[500])
    return "{:<8} -> \"{}\"  ({})".format(str(code), e["title"], e["action"])

print("Every error becomes a friendly card with a recovery action:")
for code in [401, 402, 429, 500, 503, "timeout", "network"]:
    print("  " + show(code))
print()

# Graceful degradation: a cached/smaller answer beats a blank error.
cache_has_answer = True
print("Graceful degradation on a 500 (mid-stream):")
if cache_has_answer:
    print("  keep the partial streamed text, then serve a cached answer marked 'showing a saved response'")
else:
    print("  show the friendly retry card, never a raw 500")
print("A degraded but honest answer beats a blank 'Internal Server Error' every time.")

# Output:
# Every error becomes a friendly card with a recovery action:
#   401      -> "Session expired"  (redirect to log in)
#   402      -> "Usage limit reached"  (show upgrade options)
#   429      -> "Too many requests"  (auto-retry in 5s (countdown))
#   500      -> "Our AI hit a snag"  (auto-retry in 3s, else fall back to a cached answer)
#   503      -> "High demand"  (queue the request, auto-retry in 10s)
#   timeout  -> "Taking a while"  (offer 'simplify the question')
#   network  -> "You are offline"  (auto-retry on reconnect)
#
# Graceful degradation on a 500 (mid-stream):
#   keep the partial streamed text, then serve a cached answer marked 'showing a saved response'
# A degraded but honest answer beats a blank 'Internal Server Error' every time.

- Every error code becomes a **friendly card**: 401 to log in, 402 to upgrade, 429/500/503 auto-retry with a countdown, timeout suggests simplifying, network retries on reconnect.
- The user **never sees a raw code**; transient errors often heal via auto-retry with no action needed.
- **Graceful degradation**: on a 500 mid-stream, keep the partial text and fall back to a cached answer marked “showing a saved response”.
- The lesson: a degraded but honest answer beats a blank “Internal Server Error” — this is the user-facing side of 12.9’s kill switch and 12.2’s retries.

#### 💡 What just happened

⚡ What just happened?Map every failure to a friendly message + a recovery action (auto-retry transient, redirect auth/billing, simplify on timeout), and degrade to a cached or smaller answer rather than showing nothing. Tradeoff: a fallback answer may be staler or simpler, but it beats a blank error and keeps the user moving. Next: the two patterns that close the loop - confirming risky actions and capturing feedback.

- Fire each error (401/402/429/500/503/timeout/network) and watch the raw code become a friendly card
- See the right recovery action per error: retry countdown, login, upgrade, or degrade to a cached answer

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Human-in-the-Loop and the Feedback Loop

### Human-in-the-Loop and the Feedback Loop

Risky actions - destructive, irreversible, high-cost, or low-confidence - get a confirm before they run; when the model is unsure, a clarifying question beats a confident wrong answer. And a thumbs up/down linked to each trace is where your production quality signal is born.

The last mile has two closing patterns, and together they connect the UX back to everything before it. First, **human-in-the-loop**: not every action should just run. A **destructive** (delete the records), **irreversible** (send the email), **high-cost** (place a high-cost order), or **low-confidence** action gets an explicit **confirm-before-you-act** step, while safe read-only actions run automatically. And when the model is genuinely uncertain, a short **clarifying question** beats a confident wrong answer (Horvitz’s rule) — at the moment the AI reaches its limits, hand off to the human clearly, with the full context of what it already did (this is the UX surface of the 11.10 approval graph). Second, the **feedback loop**: a **thumbs up/down** (with an optional reason) on each response, **linked to that response’s trace**, is the rawest form of the quality signal — it feeds the online eval from 12.8 and seeds the eval set from 14.4. The UX is literally where production quality data is generated. The block runs the gate and the feedback aggregation, keyless.

> ✋ **Analogy**
>
> It is the **“are you sure?” and the comment card** of a good service. A careful shop does not let you shred a signed contract without a second glance — irreversible, costly, or destructive actions get a “are you sure?” (the confirm gate), while handing you a brochure needs no ceremony (auto-run). And the comment card by the door, tied to *your* visit, is how the business actually learns what to fix — a thumbs-up/down linked to the trace is that card, feeding the kitchen’s improvements. Confirm the risky, ask when unsure, and capture the feedback: that is how the loop closes.

An agent is about to delete 40 records. What should the UX do, and what is the point of capturing thumbs feedback?

**📝 `07_hitl_feedback.py`** - *runnable*

In [ ]:
# TWO closing patterns. (1) Confirm-before-you-act on risky actions; auto-run the safe ones.
def needs_confirm(action):
    reasons = []
    if action["destructive"]:   reasons.append("destructive")
    if action["irreversible"]:  reasons.append("irreversible")
    if action["cost_usd"] > 50: reasons.append("high-cost")
    if action["confidence"] < 0.6: reasons.append("low-confidence")
    return reasons

actions = [
    {"name": "answer an FAQ",        "destructive": False, "irreversible": False, "cost_usd": 0,   "confidence": 0.95},
    {"name": "read a document",      "destructive": False, "irreversible": False, "cost_usd": 0,   "confidence": 0.9},
    {"name": "delete 40 records",    "destructive": True,  "irreversible": True,  "cost_usd": 0,   "confidence": 0.8},
    {"name": "send a customer email","destructive": False, "irreversible": True,  "cost_usd": 0,   "confidence": 0.7},
    {"name": "place a $500 order",   "destructive": False, "irreversible": True,  "cost_usd": 500, "confidence": 0.85},
]
print("Confirm-before-you-act gate:")
for a in actions:
    reasons = needs_confirm(a)
    verdict = "CONFIRM ({})".format(", ".join(reasons)) if reasons else "auto-run"
    print("  {:<24} -> {}".format(a["name"], verdict))
print("  When the model is genuinely unsure, a short clarifying question beats a confident wrong action.")
print()

# (2) The feedback loop: thumbs up/down linked to a TRACE id, aggregated into a rolling quality signal.
votes = [("trace-01", 1), ("trace-02", 1), ("trace-03", 0), ("trace-04", 1), ("trace-05", 0), ("trace-06", 1)]
print("Feedback stream (each vote linked to its trace for the 12.8 eval loop):")
running = 0
for i, (trace, v) in enumerate(votes, 1):
    running += v
    print("  {}  thumbs-{:<4} rolling up-rate {:.0%}".format(trace, "up" if v else "down", running / i))
print("These votes are where production quality data is born - they feed online eval (12.8) and the eval set (14.4).")

# Output:
# Confirm-before-you-act gate:
#   answer an FAQ            -> auto-run
#   read a document          -> auto-run
#   delete 40 records        -> CONFIRM (destructive, irreversible)
#   send a customer email    -> CONFIRM (irreversible)
#   place a $500 order       -> CONFIRM (irreversible, high-cost)
#   When the model is genuinely unsure, a short clarifying question beats a confident wrong action.
#
# Feedback stream (each vote linked to its trace for the 12.8 eval loop):
#   trace-01  thumbs-up   rolling up-rate 100%
#   trace-02  thumbs-up   rolling up-rate 100%
#   trace-03  thumbs-down rolling up-rate 67%
#   trace-04  thumbs-up   rolling up-rate 75%
#   trace-05  thumbs-down rolling up-rate 60%
#   trace-06  thumbs-up   rolling up-rate 67%
# These votes are where production quality data is born - they feed online eval (12.8) and the eval set (14.4).

**📝 `streaming-stack.py`** - *illustrative*

In [ ]:
# MODERN STREAMING STACK - an illustrative minimal example (Anthropic backend + Vercel AI SDK front-end).
import anthropic
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

client = anthropic.Anthropic()
app = FastAPI()

def sse(event: str, data: str) -> str:
    return "event: " + event + "\ndata: " + data + "\n\n"

@app.get("/v1/stream")
def stream(prompt: str):
    def gen():
        # client.messages.stream yields text deltas; forward each as an SSE token event.
        with client.messages.stream(
            model="claude-sonnet-4-6", max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
        ) as s:
            for text in s.text_stream:
                yield sse("token", text)
        yield sse("done", "{}")
    return StreamingResponse(gen(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# --- Front-end (Vercel AI SDK 5/6, React) - illustrative, not Python ---
#   const [input, setInput] = useState("");
#   const { messages, sendMessage, status, stop } = useChat();
#   // useChat inserts the user message OPTIMISTICALLY before the round-trip,
#   // streams tokens into messages, exposes status === "streaming" for a skeleton,
#   // and stop() wires the visible stop button; you submit with sendMessage(input).
#   // Beyond streaming TEXT, the SDK can stream tool/UI "parts" that the front end
#   // renders as rich components (a weather card, a form) - "generative UI".
# Output: (illustrative minimal example - needs anthropic + fastapi + a React app and a key; not run here.)

- The **confirm gate** runs safe read-only actions automatically but requires a confirm for **destructive / irreversible / high-cost / low-confidence** ones — and when unsure, asks a clarifying question.
- The **feedback stream** turns each thumbs vote into a rolling up-rate, with every vote **linked to its trace id** so it feeds the 12.8 online-eval loop and the 14.4 eval set.
- The **illustrative stack** is the current 2026 front-end: an Anthropic `client.messages.stream` backend over SSE, and the Vercel AI SDK `useChat` hook that does the optimistic insert, the streaming, and the stop button for you.
- That is the whole lesson in one surface: the UX is where fast, honest, trustworthy meets the user — and where the quality data to improve it is born.

#### 💡 What just happened

⚡ What just happened?Risky actions (destructive, irreversible, high-cost, low-confidence) get a confirm-before-you-act gate and a clarifying question when the model is unsure; and a thumbs up/down linked to each trace is where your production quality signal is born (feeding 12.8 and 14.4). Tradeoff: a confirm adds one click on the risky actions, in exchange for never auto-deleting the wrong records. That closes Module 12: the whole production system, finally, meeting the user.

- Route actions (read vs delete/email/order) to auto-run or a confirm gate
- Stream thumbs up/down into a rolling quality bar that flows to the eval loop, each vote linked to a trace

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> AI-powered UX is the last mile that turns a working system into a product people trust. It starts with **perceived performance**: stream tokens so the user reads from the first word and the wait collapses to the TTFT (Step 1). Fill the wait before that with a **skeleton, not a spinner**, matched to the length of the wait (Step 2), and make the user’s own action feel instant with **optimistic UI** that reconciles or rolls back (Step 3). Show the model’s **reasoning** in a collapsible, default-closed disclosure so it is available without being mistaken for the answer (Step 4). Be honest about doubt with **claim-level citations and a confidence signal**, flagging the unsourced claim rather than hiding it (Step 5). When things break, show a **friendly error and a recovery action**, degrading to a cached answer rather than a blank screen (Step 6). And close the loop with **human-in-the-loop confirms** on risky actions and **thumbs feedback** that feeds your evals (Step 7). Ask five questions of any AI UX: does it **feel fast**, **tell the user what is happening**, **recover gracefully**, **disclose uncertainty honestly**, and **confirm the risky actions**?

| The UX problem | Bad default | Good pattern (this lesson) |
|---|---|---|
| A long generation | a blank screen until it is done | stream tokens (perceived wait = TTFT) |
| An opaque wait | a blank spinner | a skeleton, matched to the wait |
| A slow round-trip | “did it send?” | optimistic echo + reconcile / rollback |
| A reasoning trace | dumped as the answer | collapsible “Thinking”, default closed |
| An uncertain answer | hidden uncertainty | claim-level citations + “verify this” |
| An API error | “500 Internal Server Error” | friendly message + recovery + degrade |
| A risky action | auto-execute silently | confirm-before-you-act + feedback |

> 📦 **Info**
>
> ➡️ Where this goes next — and the end of Module 12That closes **Module 12**: you can now take an AI system all the way to a human who trusts it — architecture, reliability, gateways, caching, containers, CI/CD, scaling, observability, incidents, and now the UX last mile. From here, the **cost and performance** budget behind the latency you tuned comes in Module 13; the **eval set** built from the thumbs feedback you captured — error analysis and eval-driven development — is built in Lesson 14.4. Both build on this UX layer. You already met the **trace** each feedback vote links back to in Lesson 12.8, and the human-in-the-loop **approval graph** this lesson is the UX surface of in Lesson 11.10. The primary references are the [Vercel AI SDK generative UI docs](https://ai-sdk.dev/docs/ai-sdk-ui/generative-user-interfaces), [the AI SDK source](https://github.com/vercel/ai), the [Microsoft HAX guidelines](https://www.microsoft.com/en-us/research/project/guidelines-for-human-ai-interaction/), the [Google People + AI Guidebook](https://pair.withgoogle.com/guidebook/), and [NN/g on response-time limits](https://www.nngroup.com/articles/response-times-3-important-limits/).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **AI-PoweredUX**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_10.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.10.html` - regenerate this notebook whenever the source HTML is updated.*
